In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install py_vncorenlp underthesea

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.4/978.4 kB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 52.8 MB/s eta 0:00:00
  Created wheel for py_vncorenlp: filename=py_vncorenlp-0.1.4-py3-none-any.whl size=4304 sha256=4bbe2d3e745f1a951b1932cd1cd28e85b42c0e1c74fa2d9920b60074bebb79d0
  Stored in directory: /root/.cache/pip/wheels/db/e5/ff/f4a1b4ece36e8582db1ca71150a34e987e65df50c35974e9bb
Successfully built py_vncorenlp


In [ ]:
import os

def install_java():
  !apt update -qq
  !apt-get install -y openjdk-11-jdk-headless -qq > /dev/null
  os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
  !java -version

install_java()

48 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
openjdk version "17.0.17" 2025-10-21
OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-122.04, mixed mode, sharing)


In [ ]:
!git clone https://github.com/tommachilez/virag4fc.git
%cd /content/virag4fc/

Cloning into 'fact-checking-data-framework-vn'...
remote: Enumerating objects: 260, done.
remote: Counting objects: 100% (260/260), done.
remote: Compressing objects: 100% (165/165), done.
remote: Total 260 (delta 152), reused 198 (delta 93), pack-reused 0 (from 0)
Receiving objects: 100% (260/260), 5.29 MiB | 11.06 MiB/s, done.
Resolving deltas: 100% (152/152), done.
/content/fact-checking-data-framework-vn


In [3]:
VNCORE_MODEL_PATH = "/content/drive/MyDrive/KLTN_Project/Models/vncorenlp_models"
csv_path = "/content/drive/MyDrive/KLTN_Project/Datasets/vifc.csv"
jsonl_path = "/content/drive/MyDrive/KLTN_Project/Datasets/queries_triples.jsonl"
stopwords_path = "/content/drive/MyDrive/KLTN_Project/Datasets/stopwords-vi.txt"
output_path = "/content/drive/MyDrive/KLTN_Project/Datasets/queries_lexical_scores"

# Lexical Filter

In [ ]:
!python -m src.scripts.query_lexical_filter \
    --vncorenlp {VNCORE_MODEL_PATH} \
    --csv {csv_path} \
    --jsonl {jsonl_path} \
    --stopwords {stopwords_path} \
    --output_dir {output_path} \
    --threshold 0.0 \
    # --quota 10

2025-12-12 01:31:51 INFO  WordSegmenter:24 - Loading Word Segmentation model
Loaded 610 stopwords.
Loaded 34811 documents.
Output directory set to: /content/drive/MyDrive/KLTN_Project/Datasets/queries_lexical_scores
Counting lines in JSONL file...
Processing 2692 entries with threshold 0.0...
Filtering:   0% 0/2692 [00:00<?, ?docs/s]
[DEBUG] First Entry ID in JSON: '0'
[DEBUG] ID '0' FOUND in CSV documents.
[DEBUG] Queries structure: [{"query": "Phó Thủ tướng Trần Hồng Hà chúc mừng Đài Truyền hình Việt Nam TP. Hải Phòng COVID-19", "type": "KEYWORD"}, {"query": "Phó Thủ tướng Trần Hồng Hà có chúc mừng Đài Truyền hình Việt Nam và TP...
[DEBUG] Type: keyword | Overlap: 1.00 | Threshold: 0.0
[DEBUG] Type: natural | Overlap: 1.00 | Threshold: 0.0
Filtering:   0% 1/2692 [00:01<52:48,  1.18s/docs][DEBUG] Type: keyword | Overlap: 1.00 | Threshold: 0.0
[DEBUG] Type: natural | Overlap: 1.00 | Threshold: 0.0
[DEBUG] Type: keyword | Overlap: 1.00 | Threshold: 0.0
[DEBUG] Type: natural | Overlap: 0

In [ ]:
def count_lines(file_path):
    with open(file_path, 'r') as file:
        lines = file.readlines()
    return len(lines)

display(
    count_lines(output_path + "/keyword.jsonl"),
    count_lines(output_path + "/natural.jsonl"),
    count_lines(output_path + "/semantic.jsonl")
)

2692

2692

2692

# Semantic Filter

In [4]:
!python -m src.scripts.query_semantic_filter \
  --csv {csv_path} \
  --jsonl {jsonl_path} \
  --column "document" \
  --output_dir "/content/scores" \
  --quota 100 \
  --batch_size 16

Target quota: 100. Remaining to process: 100
Loading ViRanker (namdp-ptit/ViRanker) on cuda...
tokenizer_config.json: 1.17kB [00:00, 3.95MB/s]
sentencepiece.bpe.model: 100% 5.07M/5.07M [00:01<00:00, 3.94MB/s]
tokenizer.json: 100% 17.1M/17.1M [00:00<00:00, 34.2MB/s]
special_tokens_map.json: 100% 964/964 [00:00<00:00, 7.52MB/s]
config.json: 100% 796/796 [00:00<00:00, 5.83MB/s]
2025-12-14 06:44:07.346926: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765694647.364781    1782 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765694647.370721    1782 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765694647.386401    1782 computation_placer.cc